# Evaluation

This notebook loads `test_doc_rules_based.docx`, converts it to a Python dictionary using the `parse_to_dict` function from `repo root > preprocessing > assessment_processor.py`, parses the resulting assessment dictionary with `AssessmentParser.parse(...)`, and then runs each checker only on the `full_report` sections whose section name contains that checker script filename.

Each checker has its own output cell below, so we can inspect the results methodically script by script.


In [19]:
from pathlib import Path
import importlib
import importlib.util
import json
import sys

EVALUATION_ROOT = Path.cwd().resolve()
PACKAGE_ROOT = EVALUATION_ROOT.parent
REPO_ROOT = EVALUATION_ROOT.parent.parent
PROCESSOR_SCRIPT = REPO_ROOT / "preprocessing" / "assessment_processor.py"

assert PACKAGE_ROOT.exists(), f"Could not find package folder from {EVALUATION_ROOT}"
assert PROCESSOR_SCRIPT.exists(), f"Could not find {PROCESSOR_SCRIPT}"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def fresh_project_imports(verbose: bool = True):
    importlib.invalidate_caches()
    stale_modules = [
        name for name in list(sys.modules)
        if name == "iucn_rules_checker" or name.startswith("iucn_rules_checker.")
    ]
    for module_name in sorted(stale_modules, key=len, reverse=True):
        del sys.modules[module_name]

    assessment_parser_module = importlib.import_module("iucn_rules_checker.assessment_parser")
    assessment_reviewer_module = importlib.import_module("iucn_rules_checker.assessment_reviewer")
    abbreviations_module = importlib.import_module("iucn_rules_checker.checkers.abbreviations")
    dates_module = importlib.import_module("iucn_rules_checker.checkers.dates")
    formatting_module = importlib.import_module("iucn_rules_checker.checkers.formatting")
    geography_module = importlib.import_module("iucn_rules_checker.checkers.geography")
    iucn_terms_module = importlib.import_module("iucn_rules_checker.checkers.iucn_terms")
    numbers_module = importlib.import_module("iucn_rules_checker.checkers.numbers")
    punctuation_module = importlib.import_module("iucn_rules_checker.checkers.punctuation")
    references_module = importlib.import_module("iucn_rules_checker.checkers.references")
    scientific_module = importlib.import_module("iucn_rules_checker.checkers.scientific")
    spelling_module = importlib.import_module("iucn_rules_checker.checkers.spelling")
    symbols_module = importlib.import_module("iucn_rules_checker.checkers.symbols")
    tables_module = importlib.import_module("iucn_rules_checker.checkers.tables")
    bibliography_module = importlib.import_module("iucn_rules_checker.checkers.bibliography")

    modules = {
        "AssessmentParser": assessment_parser_module.AssessmentParser,
        "IUCNAssessmentReviewer": assessment_reviewer_module.IUCNAssessmentReviewer,
        "AbbreviationChecker": abbreviations_module.AbbreviationChecker,
        "DateChecker": dates_module.DateChecker,
        "FormattingChecker": formatting_module.FormattingChecker,
        "GeographyChecker": geography_module.GeographyChecker,
        "IUCNTermsChecker": iucn_terms_module.IUCNTermsChecker,
        "NumberChecker": numbers_module.NumberChecker,
        "PunctuationChecker": punctuation_module.PunctuationChecker,
        "ReferenceChecker": references_module.ReferenceChecker,
        "ScientificNameChecker": scientific_module.ScientificNameChecker,
        "SpellingChecker": spelling_module.SpellingChecker,
        "SymbolChecker": symbols_module.SymbolChecker,
        "TableChecker": tables_module.TableChecker,
        "BibliographyChecker": bibliography_module.BibliographyChecker,
    }

    if verbose:
        print("Fresh imports ready:")
        for name in modules:
            print(f"- {name}")

    return modules


def load_parse_to_dict(verbose: bool = True):
    module_name = "preprocessing_assessment_processor_eval"
    if module_name in sys.modules:
        del sys.modules[module_name]

    spec = importlib.util.spec_from_file_location(module_name, PROCESSOR_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    assert spec.loader is not None
    spec.loader.exec_module(module)

    if verbose:
        print(f"Loaded parse_to_dict from: {PROCESSOR_SCRIPT}")

    return module.parse_to_dict


In [20]:
# Set CUSTOM_DOCX_PATH to a string or Path for your own `.docx` file.
# Leave it as None to use the bundled evaluation document.
CUSTOM_DOCX_PATH = None

DEFAULT_DOCX_PATH = EVALUATION_ROOT / "test_doc_rules_based.docx"
DOCX_PATH = Path(CUSTOM_DOCX_PATH).expanduser() if CUSTOM_DOCX_PATH else DEFAULT_DOCX_PATH

assert DOCX_PATH.exists(), f"Could not find {DOCX_PATH}"
print("DOCX path:", DOCX_PATH)


DOCX path: G:\My Drive\Documents\3. Education\7. Imperial\Modules\SWE Group Project\IUCN_Reviewer\iucn_rules_checker\evaluation\test_doc_rules_based.docx


## Run the Word-document conversion

This step loads the Word document and converts it into the structured assessment dictionary using the `parse_to_dict` function from `repo root > preprocessing > assessment_processor.py`.


In [21]:
parse_to_dict = load_parse_to_dict(verbose=False)
assessment = parse_to_dict(str(DOCX_PATH))

print(type(assessment).__name__)
print("Top-level keys:", list(assessment.keys()))
print("Title:", assessment.get("title"))
print("Block count:", len(assessment.get("blocks", [])))
print("Child count:", len(assessment.get("children", [])))


dict
Top-level keys: ['title', 'level', 'path', 'blocks', 'children', 'comments']
Title: test_doc_rules_based
Block count: 3
Child count: 1


## Parse the assessment dictionary into `full_report`

This step runs `AssessmentParser.parse(...)` so we can target individual parsed sections by their section names.


In [22]:
project_modules = fresh_project_imports(verbose=False)
parser = project_modules["AssessmentParser"]()
full_report = parser.parse(assessment)

print(type(full_report).__name__)
print("Parsed entries:", len(full_report))
print()
for index, section_name in enumerate(full_report.keys(), start=1):
    print(f"{index:>2}. {section_name}")
    if index == 15:
        break


dict
Parsed entries: 113

 1. test_doc_rules_based [paragraph 1]
 2. test_doc_rules_based [paragraph 2]
 3. test_doc_rules_based > Checkers > abbreviations.py [paragraph 1]
 4. test_doc_rules_based > Checkers > abbreviations.py [paragraph 2]
 5. test_doc_rules_based > Checkers > abbreviations.py [paragraph 3]
 6. test_doc_rules_based > Checkers > abbreviations.py [paragraph 4]
 7. test_doc_rules_based > Checkers > abbreviations.py [paragraph 5]
 8. test_doc_rules_based > Checkers > abbreviations.py [paragraph 6]
 9. test_doc_rules_based > Checkers > abbreviations.py [paragraph 7]
10. test_doc_rules_based > Checkers > abbreviations.py [paragraph 8]
11. test_doc_rules_based > Checkers > abbreviations.py [paragraph 9]
12. test_doc_rules_based > Checkers > abbreviations.py [paragraph 10]
13. test_doc_rules_based > Checkers > abbreviations.py [paragraph 11]
14. test_doc_rules_based > Checkers > abbreviations.py [paragraph 12]
15. test_doc_rules_based > Checkers > dates.py [paragraph 1]


In [23]:
CHECKER_SPECS = [
    ("abbreviations.py", "AbbreviationChecker"),
    ("dates.py", "DateChecker"),
    ("formatting.py", "FormattingChecker"),
    ("geography.py", "GeographyChecker"),
    ("iucn_terms.py", "IUCNTermsChecker"),
    ("numbers.py", "NumberChecker"),
    ("punctuation.py", "PunctuationChecker"),
    ("references.py", "ReferenceChecker"),
    ("scientific.py", "ScientificNameChecker"),
    ("spelling.py", "SpellingChecker"),
    ("symbols.py", "SymbolChecker"),
    ("tables.py", "TableChecker"),
    ("bibliography.py", "BibliographyChecker"),
]


def run_checker_for_script(script_filename: str, checker_class_name: str):
    project_modules = fresh_project_imports(verbose=False)
    checker = project_modules[checker_class_name]()
    cleanup_reviewer = project_modules["IUCNAssessmentReviewer"]()

    matching_sections = [
        (section_name, text)
        for section_name, text in full_report.items()
        if script_filename.lower() in section_name.lower()
    ]

    violations = []
    checker.begin_sweep()
    try:
        for section_item in matching_sections:
            violations.extend(checker.check_text(*section_item))
    finally:
        checker.end_sweep()

    cleaned_violations = cleanup_reviewer.clean_up_violations(violations)
    return matching_sections, cleaned_violations


def display_checker_run(script_filename: str, checker_class_name: str):
    matching_sections, violations = run_checker_for_script(script_filename, checker_class_name)
    print(f"Checker: {checker_class_name}")
    print(f"Script filter: {script_filename}")
    print(f"Matching sections: {len(matching_sections)}")
    for section_name, _ in matching_sections:
        print(f"- {section_name}")
    print()
    print(f"Violation count: {len(violations)}")
    print(json.dumps([violation.to_dict() for violation in violations], indent=2, ensure_ascii=False))
    return matching_sections, violations


## AbbreviationChecker

This cell runs `AbbreviationChecker` only on parsed sections whose section name contains `abbreviations.py`.


In [24]:
abbreviation_results = display_checker_run('abbreviations.py', 'AbbreviationChecker')


Checker: AbbreviationChecker
Script filter: abbreviations.py
Matching sections: 12
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 1]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 2]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 3]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 4]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 5]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 6]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 7]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 8]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 9]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 10]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 11]
- test_doc_rules_based > Checkers > abbreviations.py [paragraph 12]

Violation count: 28
[
  {
    "rule_class": "AbbreviationChecker",
    "rule_method": "AbbreviationChecker.ch

## DateChecker

This cell runs `DateChecker` only on parsed sections whose section name contains `dates.py`.


In [25]:
date_results = display_checker_run('dates.py', 'DateChecker')


Checker: DateChecker
Script filter: dates.py
Matching sections: 9
- test_doc_rules_based > Checkers > dates.py [paragraph 1]
- test_doc_rules_based > Checkers > dates.py [paragraph 2]
- test_doc_rules_based > Checkers > dates.py [paragraph 3]
- test_doc_rules_based > Checkers > dates.py [paragraph 4]
- test_doc_rules_based > Checkers > dates.py [paragraph 5]
- test_doc_rules_based > Checkers > dates.py [paragraph 6]
- test_doc_rules_based > Checkers > dates.py [paragraph 7]
- test_doc_rules_based > Checkers > dates.py [paragraph 8]
- test_doc_rules_based > Checkers > dates.py [paragraph 9]

Violation count: 14
[
  {
    "rule_class": "DateChecker",
    "rule_method": "DateChecker.check_ordinal_dates",
    "section_name": "test_doc_rules_based > Checkers > dates.py",
    "matched_text": "11th January",
    "matched_snippet": "completed on 11th January, another draft",
    "message": "Avoid ordinal dates; use '11 January' instead of '11th January'",
    "suggested_fix": "11 January"
  },

## FormattingChecker

This cell runs `FormattingChecker` only on parsed sections whose section name contains `formatting.py`.


In [26]:
formatting_results = display_checker_run('formatting.py', 'FormattingChecker')


Checker: FormattingChecker
Script filter: formatting.py
Matching sections: 10
- test_doc_rules_based > Checkers > formatting.py [paragraph 1]
- test_doc_rules_based > Checkers > formatting.py [paragraph 2]
- test_doc_rules_based > Checkers > formatting.py [paragraph 3]
- test_doc_rules_based > Checkers > formatting.py [paragraph 4]
- test_doc_rules_based > Checkers > formatting.py [paragraph 5]
- test_doc_rules_based > Checkers > formatting.py [paragraph 6]
- test_doc_rules_based > Checkers > formatting.py [paragraph 7]
- test_doc_rules_based > Checkers > formatting.py [paragraph 8]
- test_doc_rules_based > Checkers > formatting.py [paragraph 9]
- test_doc_rules_based > Checkers > formatting.py [paragraph 10]

Violation count: 12
[
  {
    "rule_class": "FormattingChecker",
    "rule_method": "FormattingChecker.check_genus_and_species",
    "section_name": "test_doc_rules_based > Checkers > formatting.py",
    "matched_text": "Benstonea",
    "matched_snippet": "Violations: Benstonea a

## GeographyChecker

This cell runs `GeographyChecker` only on parsed sections whose section name contains `geography.py`.


In [27]:
geography_results = display_checker_run('geography.py', 'GeographyChecker')


Checker: GeographyChecker
Script filter: geography.py
Matching sections: 6
- test_doc_rules_based > Checkers > geography.py [paragraph 1]
- test_doc_rules_based > Checkers > geography.py [paragraph 2]
- test_doc_rules_based > Checkers > geography.py [paragraph 3]
- test_doc_rules_based > Checkers > geography.py [paragraph 4]
- test_doc_rules_based > Checkers > geography.py [paragraph 5]
- test_doc_rules_based > Checkers > geography.py [paragraph 6]

Violation count: 10
[
  {
    "rule_class": "GeographyChecker",
    "rule_method": "GeographyChecker.check_country_names",
    "section_name": "test_doc_rules_based > Checkers > geography.py",
    "matched_text": "Afganistan",
    "matched_snippet": "refer to Afganistan, Argetina, and",
    "message": "Use ISO 3166 country name: 'Afghanistan' instead of 'Afganistan'",
    "suggested_fix": "Afghanistan"
  },
  {
    "rule_class": "GeographyChecker",
    "rule_method": "GeographyChecker.check_country_names",
    "section_name": "test_doc_rule

## IUCNTermsChecker

This cell runs `IUCNTermsChecker` only on parsed sections whose section name contains `iucn_terms.py`.


In [28]:
iucnterms_results = display_checker_run('iucn_terms.py', 'IUCNTermsChecker')


Checker: IUCNTermsChecker
Script filter: iucn_terms.py
Matching sections: 15
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 1]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 2]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 3]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 4]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 5]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 6]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 7]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 8]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 9]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 10]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 11]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 12]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 13]
- test_doc_rules_based > Checkers > iucn_terms.py [paragraph 14]
- test_doc_rules_based

## NumberChecker

This cell runs `NumberChecker` only on parsed sections whose section name contains `numbers.py`.


In [29]:
number_results = display_checker_run('numbers.py', 'NumberChecker')


Checker: NumberChecker
Script filter: numbers.py
Matching sections: 12
- test_doc_rules_based > Checkers > numbers.py [paragraph 1]
- test_doc_rules_based > Checkers > numbers.py [paragraph 2]
- test_doc_rules_based > Checkers > numbers.py [paragraph 3]
- test_doc_rules_based > Checkers > numbers.py [paragraph 4]
- test_doc_rules_based > Checkers > numbers.py [paragraph 5]
- test_doc_rules_based > Checkers > numbers.py [paragraph 6]
- test_doc_rules_based > Checkers > numbers.py [paragraph 7]
- test_doc_rules_based > Checkers > numbers.py [paragraph 8]
- test_doc_rules_based > Checkers > numbers.py [paragraph 9]
- test_doc_rules_based > Checkers > numbers.py [paragraph 10]
- test_doc_rules_based > Checkers > numbers.py [paragraph 11]
- test_doc_rules_based > Checkers > numbers.py [paragraph 12]

Violation count: 17
[
  {
    "rule_class": "NumberChecker",
    "rule_method": "NumberChecker.check_small_numbers",
    "section_name": "test_doc_rules_based > Checkers > numbers.py",
    "mat

## PunctuationChecker

This cell runs `PunctuationChecker` only on parsed sections whose section name contains `punctuation.py`.


In [30]:
punctuation_results = display_checker_run('punctuation.py', 'PunctuationChecker')


Checker: PunctuationChecker
Script filter: punctuation.py
Matching sections: 13
- test_doc_rules_based > Checkers > punctuation.py [paragraph 1]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 2]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 3]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 4]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 5]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 6]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 7]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 8]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 9]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 10]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 11]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 12]
- test_doc_rules_based > Checkers > punctuation.py [paragraph 13]

Violation count: 13
[
  {
    "rule_class": "PunctuationChecker",
    

## ReferenceChecker

This cell runs `ReferenceChecker` only on parsed sections whose section name contains `references.py`.


In [31]:
reference_results = display_checker_run('references.py', 'ReferenceChecker')


Checker: ReferenceChecker
Script filter: references.py
Matching sections: 3
- test_doc_rules_based > Checkers > references.py [paragraph 1]
- test_doc_rules_based > Checkers > references.py [paragraph 2]
- test_doc_rules_based > Checkers > references.py [paragraph 3]

Violation count: 3
[
  {
    "rule_class": "ReferenceChecker",
    "rule_method": "ReferenceChecker.check_citation_comma",
    "section_name": "test_doc_rules_based > Checkers > references.py",
    "matched_text": "(Smith, 2020)",
    "matched_snippet": "Examples include (Smith, 2020), (Mishra et",
    "message": "No comma between author and date: '(Smith 2020)'",
    "suggested_fix": "(Smith 2020)"
  },
  {
    "rule_class": "ReferenceChecker",
    "rule_method": "ReferenceChecker.check_citation_comma",
    "section_name": "test_doc_rules_based > Checkers > references.py",
    "matched_text": "(Mishra et al., 2015)",
    "matched_snippet": "(Smith, 2020), (Mishra et al., 2015), and (Brown",
    "message": "No comma betwe

## ScientificNameChecker

This cell runs `ScientificNameChecker` only on parsed sections whose section name contains `scientific.py`.


In [32]:
scientificname_results = display_checker_run('scientific.py', 'ScientificNameChecker')


Checker: ScientificNameChecker
Script filter: scientific.py
Matching sections: 3
- test_doc_rules_based > Checkers > scientific.py [paragraph 1]
- test_doc_rules_based > Checkers > scientific.py [paragraph 2]
- test_doc_rules_based > Checkers > scientific.py [paragraph 3]

Violation count: 4
[
  {
    "rule_class": "ScientificNameChecker",
    "rule_method": "ScientificNameChecker.check_species_abbreviations",
    "section_name": "test_doc_rules_based > Checkers > scientific.py",
    "matched_text": "spp",
    "matched_snippet": "note and spp in another,",
    "message": "Use 'spp.' (with period) for multiple species",
    "suggested_fix": "spp."
  },
  {
    "rule_class": "ScientificNameChecker",
    "rule_method": "ScientificNameChecker.check_species_abbreviations",
    "section_name": "test_doc_rules_based > Checkers > scientific.py",
    "matched_text": "spp",
    "matched_snippet": "sp and spp should also",
    "message": "Use 'spp.' (with period) for multiple species",
    "sugge

## SpellingChecker

This cell runs `SpellingChecker` only on parsed sections whose section name contains `spelling.py`.


In [33]:
spelling_results = display_checker_run('spelling.py', 'SpellingChecker')


Checker: SpellingChecker
Script filter: spelling.py
Matching sections: 3
- test_doc_rules_based > Checkers > spelling.py [paragraph 1]
- test_doc_rules_based > Checkers > spelling.py [paragraph 2]
- test_doc_rules_based > Checkers > spelling.py [paragraph 3]

Violation count: 6
[
  {
    "rule_class": "SpellingChecker",
    "rule_method": "SpellingChecker.check_text",
    "section_name": "test_doc_rules_based > Checkers > spelling.py",
    "matched_text": "color",
    "matched_snippet": "Violations: The color of the",
    "message": "Use UK spelling 'colour' instead of 'color'",
    "suggested_fix": "colour"
  },
  {
    "rule_class": "SpellingChecker",
    "rule_method": "SpellingChecker.check_text",
    "section_name": "test_doc_rules_based > Checkers > spelling.py",
    "matched_text": "center",
    "matched_snippet": "changed, the center of the",
    "message": "Use UK spelling 'centre' instead of 'center'",
    "suggested_fix": "centre"
  },
  {
    "rule_class": "SpellingChecker"

## SymbolChecker

This cell runs `SymbolChecker` only on parsed sections whose section name contains `symbols.py`.


In [34]:
symbol_results = display_checker_run('symbols.py', 'SymbolChecker')


Checker: SymbolChecker
Script filter: symbols.py
Matching sections: 18
- test_doc_rules_based > Checkers > symbols.py [paragraph 1]
- test_doc_rules_based > Checkers > symbols.py [paragraph 2]
- test_doc_rules_based > Checkers > symbols.py [paragraph 3]
- test_doc_rules_based > Checkers > symbols.py [paragraph 4]
- test_doc_rules_based > Checkers > symbols.py [paragraph 5]
- test_doc_rules_based > Checkers > symbols.py [paragraph 6]
- test_doc_rules_based > Checkers > symbols.py [paragraph 7]
- test_doc_rules_based > Checkers > symbols.py [paragraph 8]
- test_doc_rules_based > Checkers > symbols.py [paragraph 9]
- test_doc_rules_based > Checkers > symbols.py [paragraph 10]
- test_doc_rules_based > Checkers > symbols.py [paragraph 11]
- test_doc_rules_based > Checkers > symbols.py [paragraph 12]
- test_doc_rules_based > Checkers > symbols.py [paragraph 13]
- test_doc_rules_based > Checkers > symbols.py [paragraph 14]
- test_doc_rules_based > Checkers > symbols.py [paragraph 15]
- test_d

## TableChecker

This cell runs `TableChecker` only on parsed sections whose section name contains `tables.py`.


In [35]:
table_results = display_checker_run('tables.py', 'TableChecker')


Checker: TableChecker
Script filter: tables.py
Matching sections: 2
- test_doc_rules_based > Checkers > tables.py [table 1] [row 1]
- test_doc_rules_based > Checkers > tables.py [table 1] [row 2]

Violation count: 1
[
  {
    "rule_class": "AbbreviationChecker",
    "rule_method": "AbbreviationChecker.check_et_al",
    "section_name": "test_doc_rules_based > Checkers > tables.py [table 1] [row 2]",
    "matched_text": "et al.",
    "matched_snippet": "GeoCAT (Bachman et al. 2011).",
    "message": "Use italicized 'et al.'",
    "suggested_fix": "Italicized: et al."
  }
]


## BibliographyChecker

This cell runs `BibliographyChecker` only on parsed sections whose section name contains `bibliography.py`.


In [36]:
bibliography_results = display_checker_run('bibliography.py', 'BibliographyChecker')


Checker: BibliographyChecker
Script filter: bibliography.py
Matching sections: 5
- test_doc_rules_based > Checkers > bibliography.py [paragraph 1]
- test_doc_rules_based > Checkers > bibliography.py [paragraph 2]
- test_doc_rules_based > Checkers > bibliography.py [paragraph 3]
- test_doc_rules_based > Checkers > bibliography.py [paragraph 4]
- test_doc_rules_based > Checkers > bibliography.py [paragraph 5]

Violation count: 4
[
  {
    "rule_class": "BibliographyChecker",
    "rule_method": "BibliographyChecker.check_ampersand_usage",
    "section_name": "test_doc_rules_based > Checkers > bibliography.py",
    "matched_text": "&",
    "matched_snippet": "Violations: Smith & Jones 2020.",
    "message": "Use 'and' not '&' in bibliography author entries",
    "suggested_fix": "and"
  },
  {
    "rule_class": "AbbreviationChecker",
    "rule_method": "AbbreviationChecker.check_et_al",
    "section_name": "test_doc_rules_based > Checkers > bibliography.py",
    "matched_text": "et al.",
 

Checked by Brandon W - 17 April 2026. Good.